In [1]:
import sys
import torch

print(sys.executable)
print(torch.__version__)

/Users/anitanam/venvs/alignn23/bin/python
2.8.0


In [5]:
import joblib
import pandas as pd
import numpy as np

mm_model = joblib.load("data/processed/multimodal_xgb.joblib")
feature_columns = joblib.load("data/processed/multimodal_feature_columns.joblib")
descriptor_columns = joblib.load("data/processed/descriptor_feature_columns.joblib")
embedding_columns = joblib.load("data/processed/cgcnn_embedding_columns.joblib")

print(len(feature_columns), len(descriptor_columns), len(embedding_columns))

264 136 128


/Users/anitanam/venvs/alignn23/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [16:09:38] WARNING: /Users/runner/work/xgboost/xgboost/src/gbm/../common/error_msg.h:80: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  warnings.warn(smsg, UserWarning)


In [6]:
ri_desc = pd.concat(
    [
        pd.read_parquet("data/processed/ri_structure_dataset_clean.parquet")[["jid", "reduced_formula", "refractive_index"]].reset_index(drop=True),
        pd.read_parquet("data/processed/ri_final.parquet").reset_index(drop=True),
    ],
    axis=1,
)

drop_cols = [c for c in ri_desc.columns if c.startswith("eps")] + ["reduced_formula"]
ri_desc_fair = ri_desc.drop(columns=drop_cols, errors="ignore").copy()

cgcnn_emb = pd.read_parquet("data/processed/cgcnn_full/cgcnn_embeddings.parquet").drop_duplicates("jid")
fusion_base = ri_desc_fair.merge(cgcnn_emb.drop(columns=["refractive_index_cgcnn"]), on="jid", how="inner")

print(fusion_base.shape)
print(fusion_base.columns[:15].tolist())

(32341, 267)
['jid', 'refractive_index', 'refractive_index', 'optb88vdw_bandgap', 'formation_energy_peratom', 'ehull', 'density', 'MagpieData minimum Number', 'MagpieData maximum Number', 'MagpieData range Number', 'MagpieData mean Number', 'MagpieData avg_dev Number', 'MagpieData mode Number', 'MagpieData minimum MendeleevNumber', 'MagpieData maximum MendeleevNumber']


In [7]:
from pymatgen.core import Structure
from pymatgen.core.composition import Composition
from matminer.featurizers.composition import ElementProperty
from pathlib import Path

ep = ElementProperty.from_preset("magpie")
magpie_cols = ep.feature_labels()

def featurize_formula(formula: str) -> pd.Series:
    comp = Composition(formula)
    vals = ep.featurize(comp)
    return pd.Series(vals, index=magpie_cols)

def load_structure_from_cif(cif_path: str) -> Structure:
    return Structure.from_file(cif_path)

def predict_refractive_index_from_fusion_row(row_df: pd.DataFrame) -> float:
    # row_df must already contain the exact training columns
    x = row_df.reindex(columns=feature_columns)
    return float(mm_model.predict(x)[0])

/Users/anitanam/venvs/alignn23/lib/python3.9/site-packages/matminer/utils/data.py:326: UserWarning: MagpieData(impute_nan=False):
In a future release, impute_nan will be set to True by default.
                    This means that features that are missing or are NaNs for elements
                    from the data source will be replaced by the average of that value
                    over the available elements.
                    This avoids NaNs after featurization that are often replaced by
                    dataset-dependent averages.
  warnings.warn(f"{self.__class__.__name__}(impute_nan=False):\n" + IMPUTE_NAN_WARNING)


In [8]:
# Check duplicate column names
from collections import Counter

dupes = [k for k, v in Counter(fusion_base.columns).items() if v > 1]
print("Duplicate columns:", dupes)

Duplicate columns: ['refractive_index']


In [9]:
# Keep only one refractive_index column
fusion_base = fusion_base.loc[:, ~fusion_base.columns.duplicated()]

print(fusion_base.shape)
print([c for c in fusion_base.columns if c == "refractive_index"])

(32341, 266)
['refractive_index']


In [10]:
sample = fusion_base.iloc[[0]].copy()

pred = predict_refractive_index_from_fusion_row(
    sample.drop(columns=["jid", "refractive_index"])
)

print("True :", sample["refractive_index"].iloc[0])
print("Pred :", pred)

True : 8.296589660818475
Pred : 1.944955587387085


In [11]:
def predict_from_existing_jid(jid: str):

    row = fusion_base.loc[
        fusion_base["jid"] == jid
    ]

    if len(row) == 0:
        raise ValueError(f"{jid} not found")

    x = row.drop(
        columns=["jid", "refractive_index"]
    )

    pred = predict_refractive_index_from_fusion_row(x)

    return {
        "jid": jid,
        "true_refractive_index": float(
            row["refractive_index"].iloc[0]
        ),
        "predicted_refractive_index": pred,
    }

In [12]:
predict_from_existing_jid(
    fusion_base["jid"].iloc[0]
)

{'jid': 'JVASP-90856',
 'true_refractive_index': 8.296589660818475,
 'predicted_refractive_index': 1.944955587387085}

In [13]:
from pymatgen.core import Composition

def predict_from_formula_and_cif(formula: str, cif_path: str):
    # 1) formula -> Magpie descriptors
    desc = featurize_formula(formula)

    # 2) CIF -> CGCNN embedding
    emb = extract_cgcnn_embedding_from_cif(cif_path)

    # 3) combine into one row
    row = pd.concat([desc, emb]).to_frame().T

    # 4) match training schema
    row = row.reindex(columns=feature_columns)

    # 5) predict
    pred = mm_model.predict(row)[0]

    return {
        "formula": formula,
        "cif_path": cif_path,
        "predicted_refractive_index": float(pred),
    }

In [15]:
print("ri_desc_fair jid duplicates:", ri_desc_fair["jid"].duplicated().sum())
print("cgcnn_emb jid duplicates:", cgcnn_emb["jid"].duplicated().sum())

print("ri_desc_fair rows:", len(ri_desc_fair), "unique jid:", ri_desc_fair["jid"].nunique())
print("cgcnn_emb rows:", len(cgcnn_emb), "unique jid:", cgcnn_emb["jid"].nunique())

ri_desc_fair jid duplicates: 296
cgcnn_emb jid duplicates: 0
ri_desc_fair rows: 32341 unique jid: 32045
cgcnn_emb rows: 32045 unique jid: 32045


In [16]:
ri_desc_fair_u = ri_desc_fair.drop_duplicates("jid").copy()
cgcnn_emb_u = cgcnn_emb.drop_duplicates("jid").copy()

fusion_base = ri_desc_fair_u.merge(
    cgcnn_emb_u.drop(columns=["refractive_index_cgcnn"]),
    on="jid",
    how="inner"
)

print(fusion_base.shape)
print(fusion_base["jid"].duplicated().sum())

(32045, 266)
0


In [19]:
# Rebuild clean deployment table
ri_desc = pd.concat(
    [
        pd.read_parquet("data/processed/ri_structure_dataset_clean.parquet")[["jid", "reduced_formula", "refractive_index"]].reset_index(drop=True),
        pd.read_parquet("data/processed/ri_final.parquet").reset_index(drop=True),
    ],
    axis=1,
)

drop_cols = [c for c in ri_desc.columns if c.startswith("eps")] + ["reduced_formula"]
ri_desc_fair = ri_desc.drop(columns=drop_cols, errors="ignore").copy()

# IMPORTANT: drop duplicate JIDs
ri_desc_fair_u = ri_desc_fair.drop_duplicates("jid").copy()
cgcnn_emb_u = pd.read_parquet("data/processed/cgcnn_full/cgcnn_embeddings.parquet").drop_duplicates("jid")

fusion_base = ri_desc_fair_u.merge(
    cgcnn_emb_u.drop(columns=["refractive_index_cgcnn"]),
    on="jid",
    how="inner",
    validate="one_to_one",
)

print(fusion_base.shape)
print(fusion_base["jid"].duplicated().sum())
print(fusion_base.columns[:10].tolist())

(32045, 267)
0
['jid', 'refractive_index', 'refractive_index', 'optb88vdw_bandgap', 'formation_energy_peratom', 'ehull', 'density', 'MagpieData minimum Number', 'MagpieData maximum Number', 'MagpieData range Number']


In [21]:
import pandas as pd

# Keep a clean lookup table for formula + target
base_lookup = pd.read_parquet("data/processed/ri_structure_dataset_clean.parquet")[
    ["jid", "reduced_formula", "refractive_index"]
].drop_duplicates("jid").copy()

print(base_lookup.shape)
print(base_lookup.head())

(32045, 3)
           jid reduced_formula  refractive_index
0  JVASP-90856        TiCuSiAs          8.296590
1  JVASP-86097            DyB6         11.873256
2  JVASP-64906         Be2OsRu         14.027763
3  JVASP-64664         Ba4NaBi         10.828656
4  JVASP-86726         LuNi4Sn         15.273380


In [25]:
def extract_cgcnn_embedding_from_cif(cif_path: str) -> pd.Series:
    global _cgcnn_model, _cgcnn_args

    if _cgcnn_model is None:
        _cgcnn_model, _cgcnn_args = _load_cgcnn_model()

    radius = float(_cgcnn_args.get("radius", 8.0))
    dmin = float(_cgcnn_args.get("dmin", 0.0))
    step = float(_cgcnn_args.get("step", 0.2))
    max_num_nbr = int(_cgcnn_args.get("max_num_nbr", 12))

    embed_layer = _cgcnn_model.conv_to_fc
    captured = []

    def hook(module, inp, out):
        captured.append(out.detach().cpu())

    with tempfile.TemporaryDirectory() as td:
        td = Path(td)

        shutil.copy(cif_path, td / "sample.cif")

        # FIXED atom_init.json path
        shutil.copy(
            "/Users/anitanam/cgcnn/data/sample-regression/atom_init.json",
            td / "atom_init.json"
        )

        # CGCNN requires this file
        (td / "id_prop.csv").write_text("sample,0.0\n")

        dataset = CIFData(
            root_dir=str(td),
            max_num_nbr=max_num_nbr,
            radius=radius,
            dmin=dmin,
            step=step,
        )

        loader = DataLoader(
            dataset,
            batch_size=1,
            shuffle=False,
            collate_fn=collate_pool,
            num_workers=0,
        )

        handle = embed_layer.register_forward_hook(hook)

        try:
            with torch.no_grad():
                for inputs, target, cif_ids in loader:
                    captured.clear()
                    _ = _cgcnn_model(*inputs)

                    if not captured:
                        raise RuntimeError(
                            "No CGCNN embedding was captured."
                        )

                    emb = captured[0].squeeze(0).numpy()

                    return pd.Series(
                        emb,
                        index=[
                            f"cgcnn_emb_{i}"
                            for i in range(len(emb))
                        ],
                    )
        finally:
            handle.remove()

    raise RuntimeError("Embedding extraction failed.")

In [26]:
sample = base_lookup.iloc[0].copy()

pred = predict_from_formula_and_cif(
    sample["reduced_formula"],
    f"/Users/anitanam/optical_materials_ml/notebooks/data/processed/cgcnn_full/{sample['jid']}.cif"
)

print("jid:", sample["jid"])
print("formula:", sample["reduced_formula"])
print("true:", sample["refractive_index"])
print("pred:", pred)

jid: JVASP-90856
formula: TiCuSiAs
true: 8.296589660818475
pred: {'formula': 'TiCuSiAs', 'cif_path': '/Users/anitanam/optical_materials_ml/notebooks/data/processed/cgcnn_full/JVASP-90856.cif', 'predicted_refractive_index': 1.23111891746521}


In [27]:
print("Training features:", len(feature_columns))

row = build_multimodal_feature_row(
    sample["reduced_formula"],
    f"/Users/anitanam/optical_materials_ml/notebooks/data/processed/cgcnn_full/{sample['jid']}.cif"
)

print("Row shape:", row.shape)

missing = [c for c in feature_columns if c not in row.columns]
extra = [c for c in row.columns if c not in feature_columns]

print("Missing:", len(missing))
print("Extra:", len(extra))

print("First 20 missing:")
print(missing[:20])

Training features: 264
Row shape: (1, 264)
Missing: 0
Extra: 0
First 20 missing:
[]


In [28]:
jid = sample["jid"]

stored_emb = (
    cgcnn_emb
    .loc[cgcnn_emb["jid"] == jid]
    .filter(like="cgcnn_emb_")
    .iloc[0]
)

new_emb = extract_cgcnn_embedding_from_cif(
    f"/Users/anitanam/optical_materials_ml/notebooks/data/processed/cgcnn_full/{jid}.cif"
)

print("Stored embedding length:", len(stored_emb))
print("New embedding length:", len(new_emb))

print(
    "Mean abs difference:",
    np.mean(np.abs(stored_emb.values - new_emb.values))
)

print(
    "Max abs difference:",
    np.max(np.abs(stored_emb.values - new_emb.values))
)

Stored embedding length: 128
New embedding length: 128
Mean abs difference: 2.6007183e-07
Max abs difference: 9.536743e-07


In [29]:
jid = sample["jid"]

# Row from training dataset
train_row = (
    fusion_base
    .loc[fusion_base["jid"] == jid]
    .drop(columns=["jid", "refractive_index"])
    .iloc[0]
)

# Row generated by deployment
deploy_row = build_multimodal_feature_row(
    sample["reduced_formula"],
    f"/Users/anitanam/optical_materials_ml/notebooks/data/processed/cgcnn_full/{jid}.cif"
).iloc[0]

diff = np.abs(train_row.values - deploy_row.values)

print("Feature count:", len(diff))
print("Mean difference:", diff.mean())
print("Max difference:", diff.max())

largest = np.argsort(diff)[-20:]

comparison = pd.DataFrame({
    "feature": np.array(train_row.index)[largest],
    "train": train_row.values[largest],
    "deploy": deploy_row.values[largest],
    "abs_diff": diff[largest],
})

comparison.sort_values("abs_diff", ascending=False)

Feature count: 264
Mean difference: nan
Max difference: nan


,feature,train,deploy,abs_diff
14,cgcnn_emb_54,-1.965039,-1.965039,9.536743e-07
15,cgcnn_emb_4,-2.956054,-2.956055,9.536743e-07
13,cgcnn_emb_51,-1.846283,-1.846284,8.344650e-07
6,cgcnn_emb_105,-2.399887,-2.399886,7.152557e-07
7,cgcnn_emb_83,-2.114396,-2.114395,7.152557e-07
8,cgcnn_emb_18,-2.202036,-2.202036,7.152557e-07
9,cgcnn_emb_67,-2.544307,-2.544308,7.152557e-07
10,cgcnn_emb_45,-1.927845,-1.927845,7.152557e-07
11,cgcnn_emb_41,-2.164908,-2.164909,7.152557e-07
12,cgcnn_emb_109,-2.295553,-2.295552,7.152557e-07


In [30]:
bad_cols = [
    "optb88vdw_bandgap",
    "formation_energy_peratom",
    "ehull",
    "density",
]

for c in bad_cols:
    if c in feature_columns:
        print(c)

optb88vdw_bandgap
formation_energy_peratom
ehull
density


In [31]:
feature_columns_no_dft = [
    c for c in feature_columns
    if c not in {
        "optb88vdw_bandgap",
        "formation_energy_peratom",
        "ehull",
        "density",
    }
]

len(feature_columns_no_dft)

260

In [32]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# Build training table
fusion_train = fusion_base.copy()

# Remove DFT-derived features
fusion_train = fusion_train.drop(
    columns=[
        "optb88vdw_bandgap",
        "formation_energy_peratom",
        "ehull",
        "density",
    ],
    errors="ignore",
)

print(fusion_train.shape)

X = fusion_train.drop(
    columns=["jid", "refractive_index"]
)

y = fusion_train["refractive_index"]

print(X.shape)

(32045, 263)
(32045, 260)


In [34]:
print(X.shape)

print("\nNon-numeric columns:")
print(X.select_dtypes(exclude=["number"]).columns.tolist())

print("\nColumns containing NaNs:")
nan_cols = X.columns[X.isna().any()].tolist()
print(len(nan_cols))
print(nan_cols[:50])

print("\nDtypes:")
print(X.dtypes.value_counts())

(32045, 260)

Non-numeric columns:
[]

Columns containing NaNs:
0
[]

Dtypes:
float64    132
float32    128
Name: count, dtype: int64


In [35]:
bad = X.select_dtypes(exclude=["number"])

if len(bad.columns):
    print("Bad columns:")
    print(bad.columns.tolist())
else:
    print("All columns numeric")

All columns numeric


In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

deploy_model = XGBRegressor(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)

deploy_model.fit(X_train, y_train)

pred = deploy_model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("Deployable Model")
print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

AttributeError: 'DataFrame' object has no attribute 'dtype'

In [37]:
import traceback

try:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
    )

    deploy_model = XGBRegressor(
        n_estimators=800,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
    )

    deploy_model.fit(X_train, y_train)

except Exception as e:
    traceback.print_exc()

Traceback (most recent call last):
  File "/var/folders/tq/_0v5yb6s3sq0p1h0967896gw0000gn/T/ipykernel_95571/1681200859.py", line 20, in <module>
    deploy_model.fit(X_train, y_train)
  File "/Users/anitanam/venvs/alignn23/lib/python3.9/site-packages/xgboost/core.py", line 726, in inner_f
    return func(**kwargs)
  File "/Users/anitanam/venvs/alignn23/lib/python3.9/site-packages/xgboost/sklearn.py", line 1143, in fit
    train_dmatrix, evals = _wrap_evaluation_matrices(
  File "/Users/anitanam/venvs/alignn23/lib/python3.9/site-packages/xgboost/sklearn.py", line 603, in _wrap_evaluation_matrices
    train_dmatrix = create_dmatrix(
  File "/Users/anitanam/venvs/alignn23/lib/python3.9/site-packages/xgboost/sklearn.py", line 1065, in _create_dmatrix
    return QuantileDMatrix(
  File "/Users/anitanam/venvs/alignn23/lib/python3.9/site-packages/xgboost/core.py", line 726, in inner_f
    return func(**kwargs)
  File "/Users/anitanam/venvs/alignn23/lib/python3.9/site-packages/xgboost/core.py"

In [38]:
import xgboost
import sklearn

print("xgboost:", xgboost.__version__)
print("sklearn:", sklearn.__version__)

xgboost: 2.1.4
sklearn: 1.6.1


In [39]:
print(type(y))
print(y.shape)

try:
    print(y.dtype)
except Exception as e:
    print(e)

<class 'pandas.core.frame.DataFrame'>
(32045, 2)
'DataFrame' object has no attribute 'dtype'


In [40]:
y = fusion_train["refractive_index"]

# If duplicate columns somehow remain
if isinstance(y, pd.DataFrame):
    y = y.iloc[:, 0]

print(type(y))
print(y.shape)

<class 'pandas.core.series.Series'>
(32045,)


In [41]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

deploy_model = XGBRegressor(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)

deploy_model.fit(X_train, y_train)

pred = deploy_model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("Deployable Model")
print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

Deployable Model
MAE : 1.3573313239748577
RMSE: 2.0271504048310685
R²  : 0.7750004134174637


In [42]:
import joblib

joblib.dump(
    deploy_model,
    "data/processed/deployable_refractive_index_model.joblib"
)

joblib.dump(
    feature_columns_no_dft,
    "data/processed/deployable_feature_columns.joblib"
)

print("Saved deployable model.")

Saved deployable model.


In [43]:
row = build_multimodal_feature_row(
    sample["reduced_formula"],
    f"/Users/anitanam/optical_materials_ml/notebooks/data/processed/cgcnn_full/{sample['jid']}.cif"
)

row = row.reindex(columns=feature_columns_no_dft)

pred = deploy_model.predict(row)[0]

print("true:", sample["refractive_index"])
print("pred:", pred)

true: 8.296589660818475
pred: 8.501181


In [44]:
row = build_multimodal_feature_row(
    sample["reduced_formula"],
    f"/Users/anitanam/optical_materials_ml/notebooks/data/processed/cgcnn_full/{sample['jid']}.cif"
)

row = row.reindex(columns=feature_columns_no_dft)

pred = deploy_model.predict(row)[0]

print("true:", sample["refractive_index"])
print("pred:", pred)

true: 8.296589660818475
pred: 8.501181
